In [1]:
pip -q install pytest pytest-sugar

Note: you may need to restart the kernel to use updated packages.


In [2]:
# install pytest
!pip -q install pytest pytest-sugar

# move to tdd directory
from pathlib import Path

if Path.cwd().name != 'tdd':
    %mkdir tdd
    %cd tdd

%pwd

/Users/akp/Desktop/MLOps/27_April/tdd


'/Users/akp/Desktop/MLOps/27_April/tdd'

In [3]:
%rm *.py

zsh:1: no matches found: *.py


In [11]:
%%file test_math.py
import math
def test_add():
    assert 1+1 == 2

def test_multiply():
    assert 2*3 == 6

def test_sin():
    assert math.sin(0) == 0



Overwriting test_math.py


In [12]:
!python -m pytest test_math.py

Test session starts (platform: darwin, Python 3.13.9, pytest 8.4.2, pytest-sugar 1.1.1)
rootdir: /Users/akp/Desktop/MLOps/27_April/tdd
plugins: anyio-4.10.0, sugar-1.1.1, hydra-core-1.3.2
collected 3 items                                                              

 test_math.py ✓✓✓                                                100% ██████████

Results (0.01s):
       3 passed


In [ ]:
%%writefile test_model.py

import pytest
import numpy as np
import pickle
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from fastapi.testclient import TestClient
from app import app

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

def test_model_training():
    assert model is not None, "Model training failed!"

def test_model_accuracy():
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    assert accuracy > 0.8, "Model accuracy is too low!"

def test_prediction_shape():
    sample = X_test[:5]

    predictions = model.predict(sample)

    assert len(predictions) == 5, "Prediction output size mismatch!"

def test_model_prediction():
    predictions = model.predict(X_train)
    
    assert predictions.shape[0] == X_train.shape[0], "Prediction shape is incorrect!"
    assert set(np.unique(predictions)).issubset(set(np.unique(y_train))), "Invalid class labels in predictions!"

from pathlib import Path

def test_model_prediction():

    # Check model file exists
    assert Path("iris.pkl").exists(), "Model file iris.pkl not found!"

    # Load model
    with open("iris.pkl", "rb") as f:
        model = pickle.load(f)

    # Load sample Iris data
    iris = load_iris()
    X = iris.data
    y = iris.target

    # Select one sample
    sample = X[0].reshape(1, -1)

    # Predict
    prediction = model.predict(sample)

    # Check prediction shape
    assert len(prediction) == 1, "Prediction output size is incorrect!"

    # Check valid class label
    assert prediction[0] in np.unique(y), "Invalid prediction label!"




def test_wrong_input_shape_fails():
    with open("iris.pkl", "rb") as f:
        model = pickle.load(f)

    bad_input = [[5.1, 3.5]]  # only 2 features instead of 4

    with pytest.raises(ValueError):
        model.predict(bad_input)




client = TestClient(app)

# Test API root endpoint
def test_root_endpoint():

    response = client.get("/")

    assert response.status_code == 200
    assert response.json() == {
        "message": "FastAPI is running!"
    }


# Test model prediction endpoint
def test_prediction_endpoint():

    response = client.get(
        "/predict",
        params={
            "sepal_length": 5.1,
            "sepal_width": 3.5,
            "petal_length": 1.4,
            "petal_width": 0.2
        }
    )

    # Check API response status
    assert response.status_code == 200

    # Convert JSON response
    result = response.json()

    # Verify prediction field exists
    assert "prediction" in result

    # Verify valid Iris class
    assert result["prediction"] in [
        "Setosa",
        "Versicolor",
        "Virginica"
    ]


Overwriting test_model.py


In [27]:

!pytest test_model.py

Test session starts (platform: darwin, Python 3.13.9, pytest 8.4.2, pytest-sugar 1.1.1)
rootdir: /Users/akp/Desktop/MLOps/27_April
plugins: anyio-4.10.0, sugar-1.1.1, hydra-core-1.3.2
collected 5 items                                                              

 test_model.py ✓✓✓✓✓                                             100% ██████████

Results (0.80s):
       5 passed
